In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv("../data/raw/prices_raw.csv")

raw.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains,ticker
0,2021-10-01 00:00:00-04:00,403.096594,407.819860,399.589211,406.145660,129240100,0.0,0.0,0.0,SPY
1,2021-10-04 00:00:00-04:00,404.985955,405.883838,398.775534,400.908051,128570000,0.0,0.0,0.0,SPY
2,2021-10-05 00:00:00-04:00,402.404549,407.314887,401.609564,405.079529,90682500,0.0,0.0,0.0,SPY
3,2021-10-06 00:00:00-04:00,401.497274,406.968799,399.879219,406.763031,113032200,0.0,0.0,0.0,SPY
4,2021-10-07 00:00:00-04:00,410.027264,413.104388,409.849554,410.279785,72437500,0.0,0.0,0.0,SPY


In [2]:
raw.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends',
       'Stock Splits', 'Capital Gains', 'ticker'],
      dtype='object')

In [3]:
raw["Date"] = pd.to_datetime(raw["Date"], utc=True)

In [4]:
raw["Date"].dtype

datetime64[ns, UTC]

In [5]:
raw["Date"] = raw["Date"].dt.tz_localize(None)

In [6]:
raw["Date"].dtype

dtype('<M8[ns]')

In [7]:
duplicates = raw.duplicated(["ticker", "Date"]).sum()

print("Duplicate ticker-date records:", duplicates)

Duplicate ticker-date records: 0


In [8]:
raw = raw.drop_duplicates(["ticker", "Date"])

In [9]:
raw.isna().sum()

Date                0
Open                0
High                0
Low                 0
Close               0
Volume              0
Dividends           0
Stock Splits        0
Capital Gains    3759
ticker              0
dtype: int64

In [10]:
raw[raw["Close"].isna()]

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Capital Gains,ticker


In [11]:
invalid_close = raw[raw["Close"] <= 0]

print("Rows with invalid close prices:", len(invalid_close))

Rows with invalid close prices: 0


In [12]:
invalid_high_low = raw[raw["High"] < raw["Low"]]

print("Rows where High < Low:", len(invalid_high_low))

Rows where High < Low: 0


In [13]:
trading_days = raw.groupby("ticker")["Date"].nunique()

trading_days

ticker
AAPL    1253
AGG     1253
GLD     1253
JPM     1253
MSFT    1253
QQQ     1253
SPY     1253
TLT     1253
Name: Date, dtype: int64

In [14]:
clean = raw[
    ["Date", "ticker", "Open", "High", "Low", "Close", "Volume"]
].copy()

In [15]:
clean = clean.rename(columns={
    "Date": "date",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume"
})

In [16]:
clean.head()

,date,ticker,open,high,low,close,volume
0,2021-10-01 04:00:00,SPY,403.096594,407.819860,399.589211,406.145660,129240100
1,2021-10-04 04:00:00,SPY,404.985955,405.883838,398.775534,400.908051,128570000
2,2021-10-05 04:00:00,SPY,402.404549,407.314887,401.609564,405.079529,90682500
3,2021-10-06 04:00:00,SPY,401.497274,406.968799,399.879219,406.763031,113032200
4,2021-10-07 04:00:00,SPY,410.027264,413.104388,409.849554,410.279785,72437500


In [17]:
clean = clean.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

In [18]:
clean.head()

,date,ticker,open,high,low,close,volume
0,2021-10-01 04:00:00,AAPL,138.405160,139.400043,135.683881,139.136688,94639600
1,2021-10-04 04:00:00,AAPL,138.268556,138.707484,134.864521,135.713089,98322000
2,2021-10-05 04:00:00,AAPL,136.054517,138.736787,135.927714,137.634613,80861100
3,2021-10-06 04:00:00,AAPL,136.035028,138.649016,134.962114,138.502716,83221100
4,2021-10-07 04:00:00,AAPL,139.536579,140.668013,139.204956,139.760910,61732700


In [19]:
print("Rows:", len(clean))
print("Columns:", list(clean.columns))
print("Missing values:")
print(clean.isna().sum())

Rows: 10024
Columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']
Missing values:
date      0
ticker    0
open      0
high      0
low       0
close     0
volume    0
dtype: int64


In [20]:
import os

os.makedirs("../data/clean", exist_ok=True)

In [21]:
clean.to_csv(
    "../data/clean/prices_clean.csv",
    index=False
)

print("Clean dataset saved successfully!")

Clean dataset saved successfully!
